In [20]:
import os
import glob
import cv2
import pandas as pd
from deepface import DeepFace

# Especifíca la ruta de la carpeta que contiene los videos
ruta_carpeta_videos = "C:/Users/franc/Desktop/DeepVideoFER/Gestion"
ruta_temp_data = os.path.join(ruta_carpeta_videos, "temp_data")

# Asegurarse de que la carpeta temp_data exista
if not os.path.exists(ruta_temp_data):
    os.makedirs(ruta_temp_data)

# Usamos el módulo glob para buscar archivos con una extensión específica (en este caso, videos con extensión .mp4)
ruta_archivos_videos = os.path.join(ruta_carpeta_videos, '*.mp4')
lista_archivos_videos = glob.glob(ruta_archivos_videos)

# Contamos la cantidad de archivos encontrados
m = len(lista_archivos_videos)  # Número de videos en la carpeta.

# Imprimimos el resultado
print(f'La cantidad de archivos de videos en la carpeta es: {m}')

# Creamos la base de datos que almacenará los videos y sus resultados.
df_BD = pd.DataFrame(columns=['Nombre de Imagen', 'angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy', 'Video', 'Q_images'])


# Lista todos los archivos en la carpeta
archivos = os.listdir(ruta_carpeta_videos)

# Filtra solo aquellos que terminan en .mp4
videos = [archivo for archivo in archivos if archivo.endswith('.mp4')]

for video in videos:
    ruta_video = os.path.join(ruta_carpeta_videos, video)

    def getFrame(sec):
        vidcap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
        hasFrames, image = vidcap.read()
        if hasFrames:
            cv2.imwrite(os.path.join(ruta_temp_data, "image" + str(count) + ".jpg"), image)
        return hasFrames

    vidcap = cv2.VideoCapture(ruta_video)

    # Obtener los FPS del video
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    print("FPS del video:", fps)

    # Calculamos de framerate en base a los fps del video.
    frameRate = 0.25  # 1 / fps

    sec = 0
    count = 1
    success = True
    while success:
        sec = sec + frameRate
        sec = round(sec, 2)
        success = getFrame(sec)
        count = count + 1

    vidcap.release()
    cv2.destroyAllWindows()

    # Computación afectiva de los frames del video.
    ruta_archivos = os.path.join(ruta_temp_data, '*.jpg')
    lista_archivos = glob.glob(ruta_archivos)
    n = len(lista_archivos)

    print(f'La cantidad de archivos de imágenes en la carpeta es: {n}')

    data = []

    for i in range(1, n+1):
        img_path = os.path.join(ruta_temp_data, f"image{i}.jpg")
        objs = DeepFace.analyze(img_path=img_path, actions=['emotion'], enforce_detection=False)

        nombre_imagen = f"imagen{i}.jpg"
        emociones = objs[0]['emotion']

        data.append([nombre_imagen, emociones])

    df = pd.DataFrame(data, columns=['Nombre de Imagen', 'Emociones'])
    df = df.join(df['Emociones'].apply(pd.Series))
    df = df.drop('Emociones', axis=1)
    df = df.round(1)
    nuevo_orden = ['Nombre de Imagen', 'happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear']
    df = df[nuevo_orden]
    df = df.assign(Video=video.split('.')[0])  # Esto extraerá el nombre ej: "video1" de "video1.mp4"
    df = df.assign(Q_images=n)
    
    df_BD = pd.concat([df_BD, df], ignore_index=True)

    archivos = os.listdir(ruta_temp_data)
    for archivo in archivos:
        if archivo.endswith('.jpg'):
            ruta_archivo = os.path.join(ruta_temp_data, archivo)
            os.remove(ruta_archivo)

La cantidad de archivos de videos en la carpeta es: 2
FPS del video: 30.0
La cantidad de archivos de imágenes en la carpeta es: 360


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 16.59it/s]


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 19.01it/s]


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 19.24it/s]


FPS del video: 30.0
La cantidad de archivos de imágenes en la carpeta es: 151


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 20.34it/s]


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 19.96it/s]


In [30]:
import os
import glob
import cv2
import pandas as pd
from deepface import DeepFace

def analizar_emociones_videos(ruta_carpeta_videos):

    ruta_temp_data = os.path.join(ruta_carpeta_videos, "temp_data")

    if not os.path.exists(ruta_temp_data):
        os.makedirs(ruta_temp_data)

    lista_archivos_videos = glob.glob(os.path.join(ruta_carpeta_videos, '*.mp4'))
    m = len(lista_archivos_videos)

    df_BD = pd.DataFrame(columns=['Nombre de Imagen', 'angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy', 'Video', 'Q_images'])

    videos = [archivo for archivo in os.listdir(ruta_carpeta_videos) if archivo.endswith('.mp4')]

    for video in videos:
        ruta_video = os.path.join(ruta_carpeta_videos, video)

        def getFrame(sec):
            vidcap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
            hasFrames, image = vidcap.read()
            if hasFrames:
                cv2.imwrite(os.path.join(ruta_temp_data, "image" + str(count) + ".jpg"), image)
            return hasFrames

        vidcap = cv2.VideoCapture(ruta_video)
        fps = vidcap.get(cv2.CAP_PROP_FPS)
        frameRate = 0.25  # 1 / fps
        sec = 0
        count = 1
        success = True

        while success:
            sec = sec + frameRate
            sec = round(sec, 2)
            success = getFrame(sec)
            count = count + 1

        vidcap.release()
        cv2.destroyAllWindows()

        lista_archivos = glob.glob(os.path.join(ruta_temp_data, '*.jpg'))
        n = len(lista_archivos)

        data = []

        for i in range(1, n+1):
            img_path = os.path.join(ruta_temp_data, f"image{i}.jpg")
            objs = DeepFace.analyze(img_path=img_path, actions=['emotion'], enforce_detection=False)

            nombre_imagen = f"imagen{i}.jpg"
            emociones = objs[0]['emotion']

            data.append([nombre_imagen, emociones])

        df = pd.DataFrame(data, columns=['Nombre de Imagen', 'Emociones'])
        df = df.join(df['Emociones'].apply(pd.Series))
        df = df.drop('Emociones', axis=1)
        df = df.round(1)
        nuevo_orden = ['Nombre de Imagen', 'happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear']
        df = df[nuevo_orden]
        df = df.assign(Video=video.split('.')[0])
        df = df.assign(Q_images=n)
    
        df_BD = pd.concat([df_BD, df], ignore_index=True)

        archivos = os.listdir(ruta_temp_data)
        for archivo in archivos:
            if archivo.endswith('.jpg'):
                ruta_archivo = os.path.join(ruta_temp_data, archivo)
                os.remove(ruta_archivo)

    return df_BD

In [35]:
# Para usar la función:
resultado = analizar_emociones_videos("C:/Users/franc/Desktop/DeepVideoFER/Gestion")
print(resultado)

Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 21.40it/s]


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 22.67it/s]


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 17.87it/s]


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 17.02it/s]


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 20.07it/s]


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 20.01it/s]


    Nombre de Imagen  angry  disgust  fear   sad  neutral  surprise  happy  \
0        imagen1.jpg   15.3     54.2   5.8  16.8      7.7       0.1    0.1   
1        imagen2.jpg    0.0      0.0   0.8  93.8      0.1       0.0    5.3   
2        imagen3.jpg    0.3      0.0   5.1   0.5      0.5       1.7   92.0   
3        imagen4.jpg    0.0      0.0   0.1   0.0      0.0       0.0   99.8   
4        imagen5.jpg   41.1      0.0   0.5  46.1      0.0       0.0   12.3   
..               ...    ...      ...   ...   ...      ...       ...    ...   
657    imagen147.jpg    0.0      0.0   0.5   1.3     98.2       0.0    0.0   
658    imagen148.jpg    0.0      0.0   1.7   2.1     96.1       0.1    0.0   
659    imagen149.jpg    0.0      0.0   0.7   0.1     98.9       0.2    0.0   
660    imagen150.jpg    0.0      0.0   0.1  98.6      1.4       0.0    0.0   
661    imagen151.jpg    0.1      0.0   1.9  97.4      0.6       0.0    0.0   

             Video Q_images  
0            test1      360  
1  

In [36]:
def obtener_promedios(df):
    """ 
    Calcula el promedio de las emociones por video.
    
    Parámetros:
    df (DataFrame): DataFrame que contiene las emociones y el nombre del video.
    
    Retorna:
    DataFrame: Promedios de las emociones por video.
    """
    # Lista de columnas de emociones
    columnas_emociones = ['angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy']
    
    # Seleccionamos solo las columnas de emociones y 'Video'
    datos_emociones = df[['Video'] + columnas_emociones]
    
    # Agrupar por la columna 'Video' y calcular el promedio para cada emoción
    resultados_promedio = datos_emociones.groupby('Video').mean()
    
    return resultados_promedio

In [39]:
resultados2 = obtener_promedios(df_BD)
print(resultados2)

                   angry   disgust       fear        sad  neutral  surprise  \
Video                                                                         
test1           7.116944  0.759444  17.637778  12.276944  23.6075  5.369722   
videoplayback  14.362914  0.113245  14.883444  30.638411  17.8000  1.043046   

                   happy  
Video                     
test1          33.220278  
videoplayback  21.148344  


In [21]:
df

,Nombre de Imagen,happy,sad,neutral,angry,surprise,disgust,fear,Video,Q_images
0,imagen1.jpg,0.1,48.0,6.5,0.0,0.0,0.0,45.4,videoplayback,151
1,imagen2.jpg,0.1,48.0,6.5,0.0,0.0,0.0,45.4,videoplayback,151
2,imagen3.jpg,0.1,48.0,6.5,0.0,0.0,0.0,45.4,videoplayback,151
3,imagen4.jpg,0.1,48.0,6.5,0.0,0.0,0.0,45.4,videoplayback,151
4,imagen5.jpg,0.1,48.0,6.5,0.0,0.0,0.0,45.4,videoplayback,151
...,...,...,...,...,...,...,...,...,...,...
146,imagen147.jpg,0.0,1.3,98.2,0.0,0.0,0.0,0.5,videoplayback,151
147,imagen148.jpg,0.0,2.1,96.1,0.0,0.1,0.0,1.7,videoplayback,151
148,imagen149.jpg,0.0,0.1,98.9,0.0,0.2,0.0,0.7,videoplayback,151
149,imagen150.jpg,0.0,98.6,1.4,0.0,0.0,0.0,0.1,videoplayback,151


In [22]:
df_BD 

,Nombre de Imagen,angry,disgust,fear,sad,neutral,surprise,happy,Video,Q_images
0,imagen1.jpg,15.3,54.2,5.8,16.8,7.7,0.1,0.1,test1,360
1,imagen2.jpg,0.0,0.0,0.8,93.8,0.1,0.0,5.3,test1,360
2,imagen3.jpg,0.3,0.0,5.1,0.5,0.5,1.7,92.0,test1,360
3,imagen4.jpg,0.0,0.0,0.1,0.0,0.0,0.0,99.8,test1,360
4,imagen5.jpg,41.1,0.0,0.5,46.1,0.0,0.0,12.3,test1,360
...,...,...,...,...,...,...,...,...,...,...
506,imagen147.jpg,0.0,0.0,0.5,1.3,98.2,0.0,0.0,videoplayback,151
507,imagen148.jpg,0.0,0.0,1.7,2.1,96.1,0.1,0.0,videoplayback,151
508,imagen149.jpg,0.0,0.0,0.7,0.1,98.9,0.2,0.0,videoplayback,151
509,imagen150.jpg,0.0,0.0,0.1,98.6,1.4,0.0,0.0,videoplayback,151
